# Generating Ground Truth Data

In [1]:
from utils.ingest import load_faq_data

In [2]:
documents = load_faq_data()

In [3]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

Now we generate questions for the the LLM Zoomcamp FAQ. 

In [4]:
# Keep only the questions which are about llm-zoomcamp
documents_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]

In [5]:
len(documents_llm)

103

In [6]:
for document in documents_llm:
    if document['course'] != 'llm-zoomcamp':
        print('id')

In [7]:
doc = documents_llm[0]
print(doc["id"])
print(doc["course"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
llm-zoomcamp
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


Now we'll produce the "Ground Source of Truth" by forcing the LLM to produce "structured output;" we're going to tell the LLM the exact structure it needs to produce using pydantic. 

In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

In [11]:
# Prepare the documents as JSON
import json 

user_prompt = json.dumps(documents_llm)

In [12]:
#Create the messages 

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "developer", "content": user_prompt}
]

In [13]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [14]:
result = response.output_parsed

print(result)

questions=['Can I still start the LLM Zoomcamp if I found it late, or is it too late to join now?', 'When do I actually get the confirmation email after registering, or am I already in and can just begin?', 'Where do I find the latest syllabus, homework deadlines, and progress tracker for the course?', 'How am I supposed to follow the weekly workflow from the videos to homework submission?', 'If I’m on the self-paced track, can I still get a certificate, or is that only for the live cohort?']


In [15]:
for index, r in enumerate(result.questions, start=1):
    print(index, r)

1 Can I still start the LLM Zoomcamp if I found it late, or is it too late to join now?
2 When do I actually get the confirmation email after registering, or am I already in and can just begin?
3 Where do I find the latest syllabus, homework deadlines, and progress tracker for the course?
4 How am I supposed to follow the weekly workflow from the videos to homework submission?
5 If I’m on the self-paced track, can I still get a certificate, or is that only for the live cohort?


In [16]:
from utils.evaluation_utils import llm_structured

/Users/evan/Projects/llm-zoomcamp-2026-code/module_04_evaluation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [18]:
for index, question in enumerate(result.questions, start=1):
    print(index, question)

1 Do I still need to register or wait for a confirmation email before I can start the LLM Zoomcamp, or can I just jump in and do the homework?
2 If I miss a homework or the deadline passes, can I still get the certificate as long as I finish the capstone project?
3 Where do I actually find the videos, weekly materials, deadlines, and submission progress for the course?
4 Are there live office hours for every module, and how do students usually join those sessions when they happen?
5 Can I follow the course in self-paced mode and still get a certificate, or is the certificate only for the live cohort?


In [19]:
#Lets calculate the price
usage.input_tokens, usage.output_tokens

(20134, 141)

In [20]:
from utils.evaluation_utils import calc_price

cost = calc_price(usage)
cost

{'input_cost': 0.0151005,
 'output_cost': 0.0006345000000000001,
 'total_cost': 0.015735}

In [21]:
# Now we convert these questions into ground truth records: 

records = []

for question in result.questions:
    records.append({
        "question":question,
        "course":doc['course'],
        "document":doc["id"]
    })

records

[{'question': 'Do I still need to register or wait for a confirmation email before I can start the LLM Zoomcamp, or can I just jump in and do the homework?',
  'course': 'llm-zoomcamp',
  'document': '74eb249bbf'},
 {'question': 'If I miss a homework or the deadline passes, can I still get the certificate as long as I finish the capstone project?',
  'course': 'llm-zoomcamp',
  'document': '74eb249bbf'},
 {'question': 'Where do I actually find the videos, weekly materials, deadlines, and submission progress for the course?',
  'course': 'llm-zoomcamp',
  'document': '74eb249bbf'},
 {'question': 'Are there live office hours for every module, and how do students usually join those sessions when they happen?',
  'course': 'llm-zoomcamp',
  'document': '74eb249bbf'},
 {'question': 'Can I follow the course in self-paced mode and still get a certificate, or is the certificate only for the live cohort?',
  'course': 'llm-zoomcamp',
  'document': '74eb249bbf'}]

Now we're going to apply the same idea to all questions in the database


For each document, (i.e., question) we'll do the following: 
- convert it to a json object to ease processing by the LLM
- have the LLM generate a `Questions` object
- create one "ground truth record" for each question


Since this process requires api calls, we need to insure it doesn't fail if/when it gets disconnected. 

To do so, we use the helper function. 

In [22]:
from utils.evaluation_utils import llm_structured_retry

In [23]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client, 
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for question in out.questions:
        results.append({
            "question": question,
            "document": doc["id"]
        })

    return results, usage


In [24]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents_llm[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

100%|██████████| 5/5 [00:09<00:00,  1.95s/it]


The above works but it's a loop which is running things one at a time. 

Let's use parallel processing for what it's good for :D 

In [25]:
from concurrent.futures import ThreadPoolExecutor
from utils.evaluation_utils import map_progress

In [26]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents_llm, generate_ground_truth)

100%|██████████| 103/103 [00:43<00:00,  2.36it/s]


In [27]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

In [28]:
from utils.evaluation_utils import calc_price


total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07712625

Now let's create the dataframe

In [29]:
import pandas as pd 

df_ground_truth = pd.DataFrame(ground_truth)

#Persist it for later
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)